In [ ]:
import pypsa
import pandas as pd
import os
import rainflow

# -------------------------------
# Utility: Annuity Function
# -------------------------------
def annuity(n, r):
    if r > 0:
        return r / (1.0 - (1.0 + r) ** -n)
    else:
        return 1.0 / n

# -------------------------------
# Step 1: Load Network
# -------------------------------
def load_and_create_base_network():
    file_path = './elec_s_37.nc'  # 2018
    network = pypsa.Network()
    network.import_from_netcdf(file_path)
    return network

# -------------------------------
# Step 2: Add Battery Storage
# -------------------------------
def add_battery_storage(network):
    max_hours = 8
    for bus in network.buses.index:
        network.add("StorageUnit",
                    name=f"{bus} battery",
                    bus=bus,
                    carrier="batteries",
                    p_nom_extendable=True,
                    max_hours=max_hours,
                    efficiency_store=0.92,
                    efficiency_dispatch=0.92,
                    capital_cost=annuity(16, 0.05) * 81e3 * (1 + 0.021) + max_hours * annuity(16, 0.05) * 236e3,
                    cyclic_state_of_charge=True)
    return network

# -------------------------------
# Step 3: Add CO2 Constraint
# -------------------------------
def add_co2_limit(network, co2_limit=1.0):
    base_co2_emissions = 1012028560.7495946
    network.add("GlobalConstraint",
                "CO2Limit",
                carrier_attribute="co2_emissions",
                sense="<=",
                constant=co2_limit * base_co2_emissions)
    return network

# -------------------------------
# Step 4: Solve Network
# -------------------------------
#def solve_network(network):
#    solver_options = {
#        "NumericFocus": 3,
#        "ScaleFlag": 2,
#        "Method": 2,
#        "Crossover": 1, # Changed from 0 to 1 as Alex suggested
#       "Presolve": 2,
#        "AggFill": 0
#    }
#    network.optimize(solver_name='gurobi', solver_options=solver_options)
#    return network

def solve_network(network):
    solver_options = {
        "OutputFlag": 1,
        "FeasibilityTol": 1e-6,
        "Method": 2,        # Selects the algorithm to solve the linear problem (Use barrier method)
        "Crossover": 1      # Changed from 0 to 1 as Alex suggested
    }
    network.optimize(solver_name='gurobi', solver_options=solver_options)
    return network

# -------------------------------
# Step 5: Export Network to NetCDF
# -------------------------------
def export_network(network, co2_limit=1.0):
    folder = f"all_hydro_to_phs_expand_transmission/netcdf/co2_{int(co2_limit * 100)}"
    os.makedirs(folder, exist_ok=True)
    network.export_to_netcdf(f"{folder}/network.nc")

# -------------------------------
# Step 6: Export Network Tables to CSV
# -------------------------------
def export_network_to_csv(network, co2_limit=1.0):
    folder = f"all_hydro_to_phs_expand_transmission/csv/co2_{int(co2_limit * 100)}"
    os.makedirs(folder, exist_ok=True)
    network.export_to_csv_folder(folder)

# -------------------------------
# Step 7: Rescale Loads
# -------------------------------
def rescale_loads(n):
    """
    Rescales the electricity demand (load) of each country in the network 
    based on external energy consumption data (in Gtoe) converted to TWh.

    Parameters
    ----------
    n : pypsa.Network
        The PyPSA network object.

    Returns
    -------
    pypsa.Network
        The updated network with rescaled load values.
    """

    # Electricity consumption per country (in Gtoe) - from CFE or external data
    gtoe_electricity = {
    "AT0 0": 11.975,
    "BE0 0": 15.118,
    "BG0 0": 5.364,
    "CZ0 0": 12.211,
    "DE0 0": 89.121,
    "DK0 0": 0.57 * 5.929,
    "DK6 0": 0.43 * 5.929,
    "EE5 0": 1.516,
    "ES0 0": 0.98 * 39.8473, #no data for Spain from CFE, data taken from Portugal and rescaled accordingly, i.e. d_old / d_new
    "ES3 0": 0.02 * 39.8473, 
    "FI6 0": 8.476,
    "FR0 0": 70.668,
    "GB4 0": 0.98 * 51.032,
    "GB2 0": 0.02 * 51.032,
    "GR0 0": 8.002,
    "HR0 0": 3.054,
    "HU0 0": 8.583,
    "IE2 0": 4.972,
    "IT0 0": 0.96 * 49.571,
    "IT1 0": 0.04 * 49.571,
    "LT5 0": 1.512,
    "LU0 0": 1.180,
    "LV5 0": 2.320,
    "NL0 0": 19.456,
    "PL0 0": 35.345,
    "PT0 0": 7.941,
    "RO0 0": 11.551,
    "SE6 0": 14.514,
    "SI0 0": 2.510,
    "SK0 0": 4.362
}


    # Conversion factor from Gtoe to TWh
    gtoe_to_twh = 11.630 # preferred conversion
    target_twh = pd.Series(gtoe_electricity) * gtoe_to_twh  # Convert Gtoe to TWh

    # Identify country codes from the 'bus' column in the loads table
    load_country = n.loads['bus'].apply(lambda x: x.split('_')[-1])

    # Aggregate current total load per country from the model (in MWh), then convert to TWh
    load_sums_by_country = n.loads_t.p_set.sum().groupby(load_country).sum() / 1e6

    # Calculate scaling factor per country: desired TWh / current TWh
    scaling_factors = target_twh / load_sums_by_country

    # Remove any countries for which we don't have data
    scaling_factors = scaling_factors.dropna()

    # Apply the scaling factor to all load time series per country
    for load in n.loads.index:
        country = load_country[load]
        if country in scaling_factors:
            n.loads_t.p_set[load] *= scaling_factors[country]

    return n

# -------------------------------
# Step 8: Convert Hydro to Pumped Hydro
# -------------------------------
def convert_hydro_to_phs(network):
    # Apply to all hydro units, not just selected countries
    hydro_mask = network.storage_units.carrier == "hydro"

    # Update operational characteristics for all hydro units
    network.storage_units.loc[hydro_mask, "p_min_pu"] = -1
    network.storage_units.loc[hydro_mask, "efficiency_store"] = 0.9

    return network

# -------------------------------
# Step 9: Expand Transmission Capacity
# -------------------------------
def expand_transmission_capacity(network):
    network.links.loc[:, "p_nom_extendable"] = True  # Set a high nominal power for all transmission links
    
    return network
    
# -------------------------------
# Step 10: Remove Battery Units
# -------------------------------
#def remove_batteries(network):
#    battery_units = network.storage_units[network.storage_units.carrier == "batteries"].index
#    network.storage_units.drop(battery_units, inplace=True)
#    return network

# -------------------------------
# Step 11: Run Scenarios  
# -------------------------------
if __name__ == "__main__":
    co2_limits = [0]  # Example scenario

    for co2_limit in co2_limits:
        network = load_and_create_base_network()
        network = rescale_loads(network)
        network = convert_hydro_to_phs(network)
        network = expand_transmission_capacity(network)  
        network = add_battery_storage(network)
        network = add_co2_limit(network, co2_limit)
        #network.snapshots = network.snapshots[:168]  # Optional: restrict time
        network = solve_network(network)
        export_network(network, co2_limit)
        export_network_to_csv(network, co2_limit)

INFO:pypsa.networks:Applying weightings to all columns of `snapshot_weightings`
INFO:pypsa.io:Imported network elec_s_37.nc has buses, carriers, generators, lines, links, loads, storage_units
Index(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13',
       '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
       '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37',
       '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49',
       '50', '51'],
      dtype='object', name='Line')
Index(['AL0 0 battery', 'AT0 0 battery', 'BA0 0 battery', 'BE0 0 battery',
       'BG0 0 battery', 'CH0 0 battery', 'CZ0 0 battery', 'DE0 0 battery',
       'DK0 0 battery', 'DK6 0 battery', 'EE5 0 battery', 'ES0 0 battery',
       'ES3 0 battery', 'FI6 0 battery', 'FR0 0 battery', 'GB2 0 battery',
       'GB4 0 battery', 'GR0 0 battery', 'HR0 0 battery', 'HU0 0 battery',
       'IE2 0 battery', 'IT0 0 battery', 'IT1 0 battery', 'LT5 0 

Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2598771


INFO:gurobipy:Set parameter LicenseID to value 2598771


...

INFO:gurobipy: 3699068    4.7282295e+11   2.941850e+00   3.916237e+17  51885s


 3699218    4.7282309e+11   2.055598e-01   1.438268e+16  52147s


INFO:gurobipy: 3699218    4.7282309e+11   2.055598e-01   1.438268e+16  52147s


 3699340    4.7282310e+11   1.476665e-02   1.185374e+15  52353s


INFO:gurobipy: 3699340    4.7282310e+11   1.476665e-02   1.185374e+15  52353s


In [ ]:
print(len(network.storage_units))
print(len(network.snapshots))

In [ ]:
network.generators_t.p

In [ ]:
vvv

In [ ]:
import pypsa
import pandas as pd
import os
import rainflow

# -------------------------------
# Utility: Annuity Function
# -------------------------------
def annuity(n, r):
    if r > 0:
        return r / (1.0 - (1.0 + r) ** -n)
    else:
        return 1.0 / n

# -------------------------------
# Step 1: Load Network
# -------------------------------
def load_and_create_base_network():
    file_path = './elec_s_37.nc'
    network = pypsa.Network()
    network.import_from_netcdf(file_path)
    return network

# -------------------------------
# Step 2: Add Battery Storage
# -------------------------------
def add_battery_storage(network):
    max_hours = 8
    for bus in network.buses.index:
        network.add("StorageUnit",
                    name=f"{bus} battery",
                    bus=bus,
                    carrier="batteries",
                    p_nom_extendable=True,
                    max_hours=max_hours,
                    efficiency_store=0.92,
                    efficiency_dispatch=0.92,
                    capital_cost=annuity(16, 0.05) * 81e3 * (1 + 0.021) + max_hours * annuity(16, 0.05) * 236e3,
                    cyclic_state_of_charge=True)
    return network

# -------------------------------
# Step 3: Add CO2 Constraint
# -------------------------------
def add_co2_limit(network, co2_limit=1.0):
    base_co2_emissions = 1012028560.7495946
    network.add("GlobalConstraint",
                "CO2Limit",
                carrier_attribute="co2_emissions",
                sense="<=",
                constant=co2_limit * base_co2_emissions)
    return network

# -------------------------------
# Step 4: Solve Network
# -------------------------------
def solve_network(network):
    solver_options = {
        "NumericFocus": 3,
        "ScaleFlag": 2,
        "Method": 2,
        "Crossover": 0,
        "Presolve": 2,
        "AggFill": 0
    }
    network.optimize(solver_name='gurobi', solver_options=solver_options)
    return network

# -------------------------------
# Step 5: Export Network to NetCDF
# -------------------------------
def export_network(network, co2_limit=1.0):
    folder = f"norway_hydro_scaled_results/network_co2_{int(co2_limit * 100)}"
    os.makedirs(folder, exist_ok=True)
    network.export_to_netcdf(f"{folder}/network.nc")

# -------------------------------
# Step 6: Export Network Tables to CSV
# -------------------------------
def export_network_to_csv(network, co2_limit=1.0):
    folder = f"norway_hydro_scaled_results/network_co2_{int(co2_limit * 100)}"
    os.makedirs(folder, exist_ok=True)
    network.export_to_csv_folder(folder)
    
# -------------------------------
# Step 7: Rescale Loads
# -------------------------------
def rescale_loads(n):
    """
    Rescales the electricity demand (load) of each country in the network 
    based on external energy consumption data (in Gtoe) converted to TWh.

    Parameters
    ----------
    n : pypsa.Network
        The PyPSA network object.

    Returns
    -------
    pypsa.Network
        The updated network with rescaled load values.
    """

    # Electricity consumption per country (in Gtoe) - from CFE or external data
    gtoe_electricity = {
    "AT0 0": 11.975,
    "BE0 0": 15.118,
    "BG0 0": 5.364,
    "CZ0 0": 12.211,
    "DE0 0": 89.121,
    "DK0 0": 0.57 * 5.929,
    "DK6 0": 0.43 * 5.929,
    "EE5 0": 1.516,
    "ES0 0": 0.98 * 39.8473, #no data for Spain from CFE, data taken from Portugal and rescaled accordingly, i.e. d_old / d_new
    "ES3 0": 0.02 * 39.8473, 
    "FI6 0": 8.476,
    "FR0 0": 70.668,
    "GB4 0": 0.98 * 51.032,
    "GB2 0": 0.02 * 51.032,
    "GR0 0": 8.002,
    "HR0 0": 3.054,
    "HU0 0": 8.583,
    "IE2 0": 4.972,
    "IT0 0": 0.96 * 49.571,
    "IT1 0": 0.04 * 49.571,
    "LT5 0": 1.512,
    "LU0 0": 1.180,
    "LV5 0": 2.320,
    "NL0 0": 19.456,
    "PL0 0": 35.345,
    "PT0 0": 7.941,
    "RO0 0": 11.551,
    "SE6 0": 14.514,
    "SI0 0": 2.510,
    "SK0 0": 4.362
}


    # Conversion factor from Gtoe to TWh
    gtoe_to_twh = 11.630 # preferred conversion
    target_twh = pd.Series(gtoe_electricity) * gtoe_to_twh  # Convert Gtoe to TWh

    # Identify country codes from the 'bus' column in the loads table
    load_country = n.loads['bus'].apply(lambda x: x.split('_')[-1])

    # Aggregate current total load per country from the model (in MWh), then convert to TWh
    load_sums_by_country = n.loads_t.p_set.sum().groupby(load_country).sum() / 1e6

    # Calculate scaling factor per country: desired TWh / current TWh
    scaling_factors = target_twh / load_sums_by_country

    # Remove any countries for which we don't have data
    scaling_factors = scaling_factors.dropna()

    # Apply the scaling factor to all load time series per country
    for load in n.loads.index:
        country = load_country[load]
        if country in scaling_factors:
            n.loads_t.p_set[load] *= scaling_factors[country]

    return n

# -------------------------------
# Step 7.5: Scale Hydro Inflow 
# -------------------------------
def scale_hydro_inflow_all_countries(network, factor):
    """
    Scale inflow for all hydro storage units across all countries.
    Skips PHS units if needed.
    """
    for su_name, su in network.storage_units.iterrows():
        carrier = str(su.carrier).lower()
        if "phs" in carrier:
            continue  # skip PHS if desired
        # Scale inflow time series if it exists
        if hasattr(network.storage_units_t, "inflow") and su_name in network.storage_units_t.inflow:
            network.storage_units_t.inflow[su_name] *= factor
        # Optionally scale static inflow attribute
        if "inflow" in network.storage_units.columns:
            try:
                network.storage_units.loc[su_name, "inflow"] *= factor
            except Exception:
                pass

# -------------------------------
# Step 7.6: Make Hydro & PHS Extendable
# -------------------------------
def scale_hydro_and_phs(network):
    network.generators.loc[network.generators['carrier'] == 'hydro', 'p_nom_extendable'] = True
    network.storage_units.loc[network.storage_units['carrier'].str.lower() == 'phs', 'p_nom_extendable'] = True
    return network

# -------------------------------
# Step 8: Run Scenarios
# -------------------------------
if __name__ == "__main__":
    co2_limits = [0]  # Set your desired CO2 limit fraction here

    for co2_limit in co2_limits:
        network = load_and_create_base_network()
        network = rescale_loads(network)
        #scale_hydro_inflow_in_norway(network, factor=1.5)  # <-- Increase inflow in Norway
        scale_hydro_inflow_all_countries(network, factor=1.5)
        network = scale_hydro_and_phs(network)
        network = add_battery_storage(network)
        network = add_co2_limit(network, co2_limit)
        network = solve_network(network)
        export_network(network, co2_limit)
        export_network_to_csv(network, co2_limit)

In [ ]:
# -------------------------------
# Step 7.5: Scale Hydro Inflow (Norway only)
# -------------------------------
def scale_hydro_inflow_in_norway(network, factor):
    no_buses = [bus for bus in network.buses.index if bus.endswith('_NO')]
    
    for su_name, su in network.storage_units.iterrows():
        if su.bus in no_buses:
            if hasattr(su, "carrier"):
                carrier = str(su.carrier).lower()
                if "phs" in carrier:
                    continue
            if hasattr(network.storage_units_t, "inflow") and su_name in network.storage_units_t.inflow:
                network.storage_units_t.inflow[su_name] *= factor
            if "inflow" in network.storage_units.columns:
                try:
                    network.storage_units.loc[su_name, "inflow"] *= factor
                except Exception:
                    pass

In [ ]:
network.storage_units

In [ ]:
network.storage_units_t.state_of_charge

In [ ]:
# Get all battery storage units
batteries = network.storage_units

# Initialize a dictionary to store SOC results
soc_results = {}

# Loop through each battery in the network
for battery in batteries.index:
    # Get max storage duration for the battery
    max_hours = network.storage_units.loc[battery, "max_hours"]
    
    # Get the optimized nominal power capacity
    p_nom_opt = network.storage_units.loc[battery, "p_nom_opt"]
    
    # Compute the total storage capacity (MWh)
    capacity = p_nom_opt * max_hours
    
    # Get SOC values for the battery
    soc_values = network.storage_units_t.state_of_charge[battery]
    
    # Compute SOC as a fraction of capacity
    SOC = soc_values / capacity if capacity > 0 else 0  # Avoid division by zero
    
    # Store in the results dictionary
    soc_results[battery] = SOC

# Convert results to a DataFrame for better visualization
import pandas as pd
soc_df = pd.DataFrame(soc_results)

# Print the DataFrame
print(soc_df)
print(soc_df.shape)

In [ ]:
network.storage_units[network.storage_units["carrier"] == "batteries"].p_nom_opt

In [ ]:
battery_degradation = degradation_df.loc[network.storage_units["carrier"] == "batteries"]

In [ ]:
degradation_df[network.storage_units["carrier"] == "batteries"]

In [ ]:
network.storage_units[network.storage_units["carrier"] == "batteries"].p_nom_opt

In [ ]:
# Filter just the battery entries
battery_df = degradation_df[network.storage_units["carrier"] == "batteries"].copy()

# Get p_nom_opt for batteries
battery_p_nom = network.storage_units[network.storage_units["carrier"] == "batteries"].p_nom_opt

# Add p_nom_opt as a new column
battery_df["Battery Capacity (MW)"] = battery_p_nom
battery_df